# Wine Quality — Model Interpretability with SHAP

This notebook uses SHAP (SHapley Additive exPlanations) to explain the predictions of our trained Random Forest model, providing global and local interpretability.

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from sklearn.model_selection import train_test_split

# Set non-blocking plotting backend
%matplotlib inline
shap.initjs()

## 1. Load Data, Model, and Scaler

In [ ]:
# Load dataset
df = pd.read_csv("../data/processed/winequality-processed.csv")
X = df.drop(columns=["quality"])
y = df["quality"]

# Load artifacts
with open("../models/model.pkl", "rb") as f:
    model = pickle.load(f)
with open("../models/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale test set
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

## 2. Initialize SHAP Explainer
We use `TreeExplainer` since we are explaining a tree-based model (Random Forest).

In [ ]:
# Initialize TreeExplainer on Random Forest model
explainer = shap.TreeExplainer(model)

# Calculate SHAP values for test samples
# Using a subset of 200 samples for faster computation
X_test_subset = X_test_scaled.iloc[:200]
shap_values = explainer(X_test_subset)
print(f"SHAP values shape: {shap_values.shape}")

## 3. Global Interpretability: Summary Plot
The summary plot ranks features by their average impact on the model output, showing distribution and directional impact.

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_subset, show=False)
plt.title("SHAP Feature Importance Summary", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Local Interpretability: Waterfall Plot
Let's look at the explanation for a single wine prediction. The waterfall plot shows how the baseline value (mean prediction) is shifted by each feature to arrive at the final predicted quality.

In [ ]:
# Select a single sample (e.g. index 0)
plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values[0], show=False)
plt.title("SHAP Explanation for a Single Wine Prediction", fontsize=14)
plt.tight_layout()
plt.show()